GPU được kiểm tra sau khi cài dependencies pinned, bằng tiến trình Python mới ở cell smoke.
Chạy notebook từ đầu để mount Drive và cập nhật repo trước khi chạy experiment.


# 04D — Portable local / Colab setup and experiments

Default: **non-training smoke test**. Locally, use the current checkout and a pinned Python environment; set `EDGEAI_DATA_ROOT` before importing tools to use an external EdgeAI storage tree. Colab mounts Drive, clones or fast-forward pulls the GitHub repository, and installs its pinned runtime requirements. Code executes in the connected kernel's filesystem.

Upload the complete staged `EdgeAI` tree to `/content/drive/MyDrive/EdgeAI` and commit/push local code fixes before the first Colab run. Never run 00–03B to repair a missing cache. This notebook does not push code or upload data.


In [ ]:
from pathlib import Path, PureWindowsPath
import os, subprocess, sys
try:
    from google.colab import drive
except ImportError:
    drive = None
IN_COLAB = drive is not None

REPO_URL = "https://github.com/ancaranovik/Mosquito_Wingbeat_v1.git"
if IN_COLAB:
    drive.mount("/content/drive")
    assert REPO_URL.startswith("https://github.com/") and "@" not in REPO_URL
    repo_hint = os.environ.get("EDGEAI_REPO_ROOT", "/content/edge-ai")
    if PureWindowsPath(repo_hint).drive:
        raise ValueError("Colab needs a runtime checkout, not a Windows repository path")
    repo = Path(repo_hint).resolve()
    if not repo.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    else:
        origin = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=repo, text=True).strip()
        assert origin == REPO_URL, "Existing checkout has a different origin"
        assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=repo, text=True).strip(), "Preserve uncommitted runtime changes before pulling"
        subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    hint = Path(os.environ.get("EDGEAI_REPO_ROOT", Path.cwd())).resolve()
    repo = next((p for p in [hint, *hint.parents] if (p / "tools/project_paths.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Set EDGEAI_REPO_ROOT to your local checkout")
os.environ["EDGEAI_REPO_ROOT"] = str(repo)
os.chdir(repo)
sys.path.insert(0, str(repo / "tools"))
sys.dont_write_bytecode = True
print("Mode:", "COLAB" if IN_COLAB else "LOCAL")
print("Checkout:", repo)


In Colab, install the pinned cache-consumer and notebook dependencies. Locally, use the already configured environment without installing packages. Verification and smoke testing use fresh child Python processes. Configure private GitHub authentication interactively; never embed credentials here.


In [ ]:
from colab_bootstrap import configure
paths = configure(install=IN_COLAB)


Kiểm tra full frozen identities và baseline đã lưu; không train hoặc tạo lại dữ liệu.
Có thể bỏ cell này khi fixed inputs đã được audit PASS và không thay đổi.
GPU smoke/preflight ở cell tiếp theo vẫn cần chạy; không bỏ hash gate.


In [ ]:
subprocess.run([sys.executable, "-B", "tools/verify_portability.py"], cwd=repo, check=True)


Colab requires CUDA and reports the GPU name before eight synthetic forward checks. Local mode uses CPU. Neither mode constructs an optimizer or starts training.


In [ ]:
device = "cuda" if IN_COLAB else "cpu"
subprocess.run([sys.executable, "-B", "tools/experiment_runner.py", "smoke", "--device", device], cwd=repo, check=True)
print("Future outputs:", paths.EXPERIMENT_ROOT / "<experiment_id>")


## Lần tiếp theo: exp_008_power075_wd1e4 — Adam L2, đủ 8 cấu hình

Chỉ đổi Adam `weight_decay` từ **0** (exp007) lên **0.0001**.
Giữ weighted CE exponent **0.75**, Adam LR=0.001, train_global_zscore TRAIN-only,
seed 42, batch 64, max 50 epoch, patience 8, split/cache/kiến trúc.
Protocol v5; engine baseline/v2/v3/v4 giữ nguyên. Không augmentation hoặc tạo lại feature.
Adam weight_decay là L2 trong optimizer; đây không phải AdamW.

Chọn checkpoint theo validation Macro-F1; early stopping vẫn theo weighted validation loss.
**TEST tắt**. Đối chứng chính **exp_007_power075**; 04C có thể xem thêm exp004.
Đánh giá window F1, recall/F1 từng loài, source F1 và khoảng cách TRAIN–VAL.
Regularization là giả thuyết cần kiểm tra, không bảo đảm F1 tăng.

Preview phải in `weight_decay=0.0001`, exponent 0.75, protocol v5 và đủ 8 run.
Tên/mô tả chỉnh tại cell dưới; params nằm trong JSON config.
Commit/push với RUN_TRAINING=False. Sau setup + GPU smoke PASS và preview đúng,
đổi True trong bản điều khiển; không cần push True.
Checkout Colab phải sạch: nếu chỉnh/lưu notebook trong repo, dùng bản sao ngoài
`/content/edge-ai` trước khi bật train. Code cũ đã import: pull rồi restart kernel.

Một lần chạy = 4 model 03A rồi 4 model 03B; kết quả mới:
`MyDrive/EdgeAI/experiments/exp_008_power075_wd1e4/`.
Tên đã tồn tại bị từ chối; retry dùng tên mới như exp_008_power075_wd1e4_retry01.
Không cần chạy lại 00–03B hoặc train riêng 04A/04B.
Sau COMPLETE 8/8, mở 04C cùng tên để xem histories/confusion matrices.
Output exp007 trước khi cập nhật notebook đã được sao lưu cục bộ tại
`_local_only/notebook_snapshots/before_exp008_20261002_123046`.
Hướng dẫn chi tiết: `docs/EXP_008_WEIGHT_DECAY.md`.


In [ ]:
# CHỈNH ĐIỀU KHIỂN TẠI ĐÂY; mỗi lần train dùng tên chưa tồn tại.
EXPERIMENT_NAME = "exp_008_power075_wd1e4"
EXPERIMENT_DESCRIPTION = "Lần 8: single-factor Adam L2 ablation versus exp_007_power075: weight_decay 0 -> 0.0001; keep weighted CE exponent 0.75, Adam LR=0.001, TRAIN-only global z-score, seed 42, batch 64, max 50 epochs, patience 8, eight frozen frontend/model pairs, split/cache, validation Macro-F1 selection and weighted validation-loss stopping. No augmentation or feature regeneration. Validation-only; TEST disabled."
EXPERIMENT_CONFIG = "configs/experiments/power075_wd1e4_v5.json"
RUN_TRAINING = False

from cache_consumer import read_json
import experiment_runner
from experiment_runner import run_plan, training_module
config_preview = read_json(repo / EXPERIMENT_CONFIG)
config_preview.update(experiment_id=EXPERIMENT_NAME, description=EXPERIMENT_DESCRIPTION)
protocol = config_preview.get("protocol_version")
if protocol and protocol not in getattr(experiment_runner, "VERSIONED_PROTOCOLS", ()):
    raise RuntimeError(
        f"Loaded runner does not support {protocol}. Pull the latest repo, restart the kernel, "
        "then rerun setup and preview with RUN_TRAINING=False. "
        f"Loaded module: {experiment_runner.__file__}"
    )
plan = run_plan(config_preview)
engine = training_module(config_preview)
effective = (engine.effective_config(config_preview["training_overrides"], config_preview["evaluate_test"])
             if "protocol_version" in config_preview else engine.CONFIG)
experiment_path = paths.experiment(EXPERIMENT_NAME)
print("Experiment:", EXPERIMENT_NAME)
print("Kết quả:", experiment_path)
print("Reference:", config_preview.get("reference_experiment_id", "accepted baseline"))
print("Additional comparison in 04C:", "exp_004_train_norm")
print("Params thực thi:", effective)
if hasattr(engine, "class_weight_record"):
    from cache_consumer import CLASSES, COUNTS, TRAIN_COUNTS
    accepted_weights = {label: COUNTS["train"] / (4 * count)
                        for label, count in zip(CLASSES, TRAIN_COUNTS)}
    print("TRAIN class weights:", engine.class_weight_record(effective, accepted_weights))
print("Kế hoạch:", len(plan), "cấu hình, chạy lần lượt:", [p["run_id"] for p in plan])
if RUN_TRAINING:
    subprocess.run([
        sys.executable, "-u", "-B", "tools/experiment_runner.py", "train",
        "--config", EXPERIMENT_CONFIG, "--experiment-id", EXPERIMENT_NAME,
        "--description", EXPERIMENT_DESCRIPTION,
    ], cwd=repo, check=True)
else:
    print("Training disabled. Kiểm tra tên/config/params, rồi đổi RUN_TRAINING thành True khi muốn chạy.")


## Xem kết quả sau khi đủ 8 cấu hình hoàn tất

Cell này chỉ đọc/xác thực kết quả; không train hoặc chạy lại TEST. Trong 04C, đặt cùng `EXPERIMENT_NAME` để xem history và confusion matrix của cả 8 cấu hình.


In [ ]:
if RUN_TRAINING:
    from experiment_review import compare_saved_results
    from IPython.display import display
    display(compare_saved_results(EXPERIMENT_NAME))
else:
    print("Để xem lần đã chạy, mở 04C và nhập EXPERIMENT_NAME tương ứng; không cần bật training.")
